In [282]:
import requests
import pandas as pd
import json
from upsetplot import UpSet, from_contents
import itertools
from functools import reduce
from pathlib import Path
import mygene
from io import StringIO
import time
import numpy as np

In [283]:
# the query protein
query = "CADM4_HUMAN"

In [284]:
# Function that adds '_HUMAN' afterfix values

def add_suffix_human(value):
    return value + '_HUMAN'

In [285]:
# Function that takes a ppi dataframe and outputs from two columns only one column with the interactor of a specified single protein

def get_interactors_for_target(df, col_a, col_b, target_protein):    
    def get_interactors(row):
        if row[col_a] == target_protein:
            return row[col_b]
        elif row[col_b] == target_protein:
            return row[col_a]
        else:
            return None
    df['interactor_of_' + target_protein] = df.apply(get_interactors, axis = 1)
    
    return df

In [286]:
# Function that uses the UniProt-API to convert the uniprotID to Protein name or vice versa (not in batch retrieval)

def convert_uniprotID_uniprotName(uniprotID_or_name): # e.g. can be P19320 or VCAM1_HUMAN

    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{uniprotID_or_name}?format={format}"
    
    try:
        uniprot_response = requests.get(uniprot_request_url)
        uniprot_response.raise_for_status()
        
        uniprot_json = uniprot_response.text
        uniprot_dict = json.loads(uniprot_json) # convering to dictionary

        if "_" in uniprotID_or_name:
            value = uniprot_dict["primaryAccession"]
        else:
            value = uniprot_dict['uniProtkbId']
        
        return (value)
    
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        
    except json.JSONDecodeError as e:
        print(f"JSON decoding error: {e}")
        
    return None

# example
id_or_name = convert_uniprotID_uniprotName("Q61490")
print(id_or_name)

CD166_MOUSE


In [301]:
# ID mapping from the Uniprot-API is used to batch retrieve uniprotIDs or uniprotNames

# this function extracts only the main protein & not the isoforms from a list of uniprot isoform proteins, which is used in the next function

def extract_main_protein(isoform_list):

    convert_None_to_NaN = [np.nan if isoform is None else isoform for isoform in isoform_list] # convert None to NaN
    convert_float_to_str = list(str(isoform) if isoform is np.nan else isoform for isoform in convert_None_to_NaN) # conversion of NaN floats to NaN strings
    protein_list = list(filter(lambda isoform: "-" not in isoform, convert_float_to_str)) # the iteration, filtering out the isoforms
    
    return protein_list

# example
input_list = ["L1CAM_MOUSE", "L1CAM_HUMAN", "CALM_DROME", None]

test = extract_main_protein(input_list)
print("test:", test)



# the actual function 

def convert_uniprotID_uniprotAcNr_from_df_column(df, column_to_convert, new_column_name):
    
    """ Converts uniprot IDs to uniprot accession numbers and vice versa from a df column 
        through batch retrieval.
    
    Converts a pandas dataframe column that contains uniprot accession numbers (e.g. Q13740) 
    to uniprot IDs (e.g. CD166_HUMAN) & vice versa. It uses the Uniprot-API ID mapping
    and does this in batch retrieval so it has one large payload for in POST request.
    
    Args:
        df: the dataframe that contains the column that needs to be converted.
        columnd_to_convert: the column in the dataframe that needs conversion, 
            this column holds either uniprot IDs or accession numbers.
        new_column_name: name of the new column where the converted items 
            will be stored.
    
    Returns:
        An exact copy of the original dataframe but it holds an extra column that 
        contains the converted items, either uniprot IDs or names. It will also print
        the job ID so that you can check with Postman for example if it is not returning
        the expected output.
    
    Raises:
        JOB status error: job status is not running & will not get results, thus something 
            went wrong with job
        ValueError: The returned list from UniProt-API is not equal in length to the nr of 
            rows in the original dataframe
    """
    
    # Conversions for input into Uniprot-API
    df[column_to_convert] = df[column_to_convert].astype(str) # making sure it is a string
    list_of_uniprotIDs_or_Names = df[column_to_convert].tolist() # column to list
    ls = ",".join(list_of_uniprotIDs_or_Names) # conversion to input as payload to uniprot
    
    # making the POST request
    r = requests.post("https://rest.uniprot.org/idmapping/run", data={
        "from": "UniProtKB_AC-ID",
        "to": "UniProtKB-Swiss-Prot", 
        "ids": ls 
    })

    # getting the job ID nr
    job_id = r.json()["jobId"]
    print("job ID:", job_id)

    # for loop that checks whether the job is running and when job is finished it will return a list
    while True:
        response = requests.get(f"https://rest.uniprot.org/idmapping/status/{job_id}")
        data_json = json.loads(response.text)
    
        if "jobStatus" in data_json:
            job_status = data_json["jobStatus"]
            print(f"job status: {job_status}")
            
            if job_status != "RUNNING":
                print("JobError: job status is not running, error with posting the request")
                break
    
        if "results" in data_json:
            if "_" in ls:
                desired_id = []
                for entry in list_of_uniprotIDs_or_Names:
                    matched = False
                    for json_entry in data_json["results"]:
                        if entry == json_entry["from"]:
                            desired_id.append(json_entry["to"]["primaryAccession"])
                            matched = True
                            break
                    if not matched:
                        desired_id.append("nan")
                desired_id_proteins = extract_main_protein(desired_id)
                break

            else:
                desired_id_proteins = []
                for entry in list_of_uniprotIDs_or_Names:
                    matched = False
                    for json_entry in data_json["results"]:
                        if entry == json_entry["from"]:  
                            desired_id_proteins.append(json_entry["to"]["uniProtkbId"])
                            matched = True
                            break
                    if not matched:
                        desired_id_proteins.append("nan")
                break
        
        time.sleep(5) # wait for 5sec
    
    # check before appending list to original dataframe, if the list has the same nr of items as the original df has rows
    if len(desired_id_proteins) != len(list_of_uniprotIDs_or_Names):
        raise ValueError("Data appending conflict: the returned list from UniProt-API is not equal in length to the nr of rows in the original dataframe")

    # now append list to dataframe   
    df[new_column_name] = desired_id_proteins
    
    return df


# testing the function
# dummy dataframe
dummy_data = {'Name':['Karan','Rohit','Sahil','Aryan'],'Protein':["Q71RJ2", "None", "P97836", None]}
dummy_df = pd.DataFrame(dummy_data)

# testing
test = convert_uniprotID_uniprotAcNr_from_df_column(dummy_df, "Protein", "ProteinID")
test.head(5)


test: ['L1CAM_MOUSE', 'L1CAM_HUMAN', 'CALM_DROME', 'nan']
job ID: 825dc65a721ead0402beb7d7c7b21560849a23cf


,Name,Protein,ProteinID
0,Karan,Q71RJ2,CCG2_RAT
1,Rohit,None,nan
2,Sahil,P97836,DLGP1_RAT
3,Aryan,None,nan


In [289]:
# this function will convert uniprotName to uniprotID

def convert_uniprotName_to_uniprotID(uniprotName):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
    fields = "accession,id"
    format = "tsv"
    request_url = f"{uniprot_api_url}{uniprotName}&fields={fields}&format={format}"
    
    try:
        uniprot_response = requests.get(request_url)
        uniprot_response.raise_for_status()
        
        tsv_data = uniprot_response.text
        df = pd.read_csv(StringIO(tsv_data), sep='\t')
        filtered_df = df[df["Entry Name"] == uniprotName]
        uniprotID = filtered_df.iloc[0,0]
    
        return (uniprotID)

    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
    
    return None

# example
l = convert_uniprotName_to_uniprotID("ITB1_HUMAN")
print(l)



P05556


In [290]:
# this function will convert uniprotID to uniprotName

def convert_uniprotID_to_uniprotName(uniprotID):

    if pd.isna(uniprotID):
        return None
    
    else:
        uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
        fields = "accession,id"
        format = "tsv"
        request_url = f"{uniprot_api_url}{uniprotID}&fields={fields}&format={format}"
        
        try:
            uniprot_response = requests.get(request_url)
            uniprot_response.raise_for_status()
            
            tsv_data = uniprot_response.text
            df = pd.read_csv(StringIO(tsv_data), sep='\t')
            filtered_df = df[df["Entry"] == uniprotID]
            uniprotName = filtered_df.iloc[0,1]
        
            return (uniprotName)
        
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
        
        return None

# example
l = convert_uniprotID_to_uniprotName("P05556")
print(l)

ITB1_HUMAN


In [291]:
# Function that uses the MyGene-API to convert the entrezGeneID to UniprotID (this is for biogrid, because biogrid outputs only gene names)

def convert_geneID_uniprotID(geneID):

    mygene_api_url = "https://mygene.info/v3/gene"
    entrezGeneID = geneID
    mygene_request_url = f"{mygene_api_url}/{entrezGeneID}?fields=all&dotfield=false&size=10"

    mygene_response = requests.get(mygene_request_url)
    mygene_json = mygene_response.json()

    if "uniprot" in mygene_json:
        uniprot_id = mygene_json["uniprot"]["Swiss-Prot"]
    elif "pantherdb" in mygene_json:
        uniprot_id = mygene_json["pantherdb"]["uniprot_kb"]
    else:
        uniprot_id = "NA"
        
    return uniprot_id

# example
Uniprot_convert = convert_geneID_uniprotID(6047)
print(Uniprot_convert)

P78317


In [292]:
# Function that uses the MyGene-API python package to convert multiple entrezGeneIDs to UniprotIDs from a dataframe column (this is for biogrid)
# This is an alternative to the function above, this uses the MyGene package instead querying the each input seperately through the API

def convert_geneIDs_uniprotIDs(df, df_column_name, df_new_column_name):

    mg = mygene.MyGeneInfo()
    
    # convert column to list
    list_of_column = df[df_column_name].tolist()
    
    # get the data from MyGene-API package
    df_mg = mg.getgenes(list_of_column, fields = 'uniprot', as_dataframe = True)
    
    # filter it, convert to list & add column to original df
    df_mg_filtered = df_mg[['uniprot.Swiss-Prot']]
    list_filtered = df_mg_filtered['uniprot.Swiss-Prot'].tolist()
    
    # append the list to the dataframe
    df[df_new_column_name] = list_filtered
    
    return df

In [293]:
# Function that uses the UniProt API to get the protein Name from the string ID (this is for string db)

def convert_stringID_to_uniprotName(string_id):
  
  uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query=gene_exact:"

  query = string_id

  uniprot_request_url = f"{uniprot_api_url}{query}+AND+organism_id:9606"

  uniprot_response = requests.get(uniprot_request_url)
  uniprot_json = uniprot_response.text
  uniprot_dict = json.loads(uniprot_json)

  if "results" in uniprot_dict and len(uniprot_dict["results"]) > 0:
      uniprot_name = uniprot_dict["results"][0]["uniProtkbId"]
  else:
      uniprot_name = None

  return uniprot_name


# example
l = convert_stringID_to_uniprotName("HAVCR1")
print(l)


HAVR1_HUMAN


In [294]:
# If you have a df with duplicate values for a certain column but not duplicate value for the other columns, this function will make of these rows, one row but retaining the info

def removeDuplicateRow_butRetainInfo (df, rowWithDuplicate, columnRetain1, columnRetain2, columnRetain3, columnRetain4 = None):
    
    def join_columnValues(series):
        return ', '.join(str(value) for value in series)

    columns_to_aggregate = {
        columnRetain1: join_columnValues,
        columnRetain2: join_columnValues,
        columnRetain3: join_columnValues
        }
    
    if columnRetain4 is not None:
        columns_to_aggregate[columnRetain4] = join_columnValues
        
    df_final = df.groupby(rowWithDuplicate).agg(columns_to_aggregate).reset_index()
    
    return df_final

In [295]:
# make a function for putting the data into the right format for the upsetplot (at the end of the script)

def convert_column_to_list(df, column):
    
    column_list = df[[column]].values.tolist()
    
    def flatten_list(nested_list):
        return list(itertools.chain(*nested_list))

    interactors = flatten_list(column_list)

    return interactors

In [304]:
# Extracting the data from the BioGRID API

# BioGRID Access Key: caf14dfd9a0b7be447d282c322b8362e

biogrid_api_url = "https://webservice.thebiogrid.org/interactions"

geneList = [convert_uniprotID_uniprotName(query)]

params = {
    "accesskey": "caf14dfd9a0b7be447d282c322b8362e", # need to request
    "additionalIdentifierTypes": "UNIPROT",
    "format": "json",
    "geneList": geneList,
    "taxId": 9606, # human
    "max": 100000
}

response = requests.get(biogrid_api_url, params=params)
biogrid_interactions = response.json()
print(response)
print(biogrid_interactions)

biogrid_data = {}
for interaction_id, interaction in biogrid_interactions.items():
    biogrid_data[interaction_id] = interaction
    biogrid_data[interaction_id]["INTERACTION_ID"] = interaction_id
    
# loading into dataframe

biogrid_df = pd.DataFrame.from_dict(biogrid_data, orient="index")

columns = [
    "INTERACTION_ID",
    "ENTREZ_GENE_A",
    "ENTREZ_GENE_B",
    "OFFICIAL_SYMBOL_A",
    "OFFICIAL_SYMBOL_B",
    "EXPERIMENTAL_SYSTEM",
    "PUBMED_ID",
    "PUBMED_AUTHOR",
    "THROUGHPUT",
    "QUALIFICATIONS"]

biogrid_df = biogrid_df[columns]

biogrid_df.head(5)

<Response [200]>
{'600990': {'BIOGRID_INTERACTION_ID': 600990, 'ENTREZ_GENE_A': '1454', 'ENTREZ_GENE_B': '199731', 'BIOGRID_ID_A': 107838, 'BIOGRID_ID_B': 128268, 'SYSTEMATIC_NAME_A': 'RP1-5O6.1', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'CSNK1E', 'OFFICIAL_SYMBOL_B': 'CADM4', 'SYNONYMS_A': 'CKIepsilon|HCKIE', 'SYNONYMS_B': 'IGSF4C|NECL4|Necl-4|TSLL2|synCAM4', 'EXPERIMENTAL_SYSTEM': 'Two-hybrid', 'EXPERIMENTAL_SYSTEM_TYPE': 'physical', 'PUBMED_AUTHOR': 'Vinayagam A (2011)', 'PUBMED_ID': 21900206, 'ORGANISM_A': 9606, 'ORGANISM_B': 9606, 'THROUGHPUT': 'High Throughput', 'QUANTITATION': '-', 'MODIFICATION': '-', 'ONTOLOGY_TERMS': {}, 'QUALIFICATIONS': '-', 'TAGS': '-', 'SOURCEDB': 'BIOGRID'}, '601286': {'BIOGRID_INTERACTION_ID': 601286, 'ENTREZ_GENE_A': '1780', 'ENTREZ_GENE_B': '199731', 'BIOGRID_ID_A': 108118, 'BIOGRID_ID_B': 128268, 'SYSTEMATIC_NAME_A': '-', 'SYSTEMATIC_NAME_B': '-', 'OFFICIAL_SYMBOL_A': 'DYNC1I1', 'OFFICIAL_SYMBOL_B': 'CADM4', 'SYNONYMS_A': 'DNCI1|DNCIC1', 'SYNON

,INTERACTION_ID,ENTREZ_GENE_A,ENTREZ_GENE_B,OFFICIAL_SYMBOL_A,OFFICIAL_SYMBOL_B,EXPERIMENTAL_SYSTEM,PUBMED_ID,PUBMED_AUTHOR,THROUGHPUT,QUALIFICATIONS
600990,600990,1454,199731,CSNK1E,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-
601286,601286,1780,199731,DYNC1I1,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-
1051917,1051917,6472,199731,SHMT2,CADM4,Affinity Capture-RNA,22658674,Castello A (2012),High Throughput,-
1196008,1196008,253012,199731,HEPACAM2,CADM4,Affinity Capture-MS,26186194,Huttlin EL (2015),High Throughput,BioPlex 1.0 HEK 293T cells CompPASS score = 0....
2232293,2232293,29785,199731,CYP2S1,CADM4,Affinity Capture-MS,28514442,Huttlin EL (2017),High Throughput,BioPlex 2.0 HEK 293T cells CompPASS score = 0....


In [305]:
# Converting EntrezGeneIDs to UniprotAccessionNumbers using batch retrieval
biogrid_df_filter = convert_geneIDs_uniprotIDs(biogrid_df, 'ENTREZ_GENE_A', 'UniprotID_A')
biogrid_df_filter2 = convert_geneIDs_uniprotIDs(biogrid_df, 'ENTREZ_GENE_B', 'UniprotID_B')
biogrid_df_filter2.head(5)

querying 1-51...done.
querying 1-51...done.


,INTERACTION_ID,ENTREZ_GENE_A,ENTREZ_GENE_B,OFFICIAL_SYMBOL_A,OFFICIAL_SYMBOL_B,EXPERIMENTAL_SYSTEM,PUBMED_ID,PUBMED_AUTHOR,THROUGHPUT,QUALIFICATIONS,UniprotID_A,UniprotID_B
600990,600990,1454,199731,CSNK1E,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-,P49674,Q8NFZ8
601286,601286,1780,199731,DYNC1I1,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-,O14576,Q8NFZ8
1051917,1051917,6472,199731,SHMT2,CADM4,Affinity Capture-RNA,22658674,Castello A (2012),High Throughput,-,P34897,Q8NFZ8
1196008,1196008,253012,199731,HEPACAM2,CADM4,Affinity Capture-MS,26186194,Huttlin EL (2015),High Throughput,BioPlex 1.0 HEK 293T cells CompPASS score = 0....,A8MVW5,Q8NFZ8
2232293,2232293,29785,199731,CYP2S1,CADM4,Affinity Capture-MS,28514442,Huttlin EL (2017),High Throughput,BioPlex 2.0 HEK 293T cells CompPASS score = 0....,Q96SQ9,Q8NFZ8


In [306]:
# Converting the UniprotID to Uniprot Name using batch retrieval
tpp = convert_uniprotID_uniprotAcNr_from_df_column(biogrid_df_filter2, "UniprotID_A", "UniprotName_A")
ttp = convert_uniprotID_uniprotAcNr_from_df_column(tpp, "UniprotID_B", "UniprotName_B")
tpp.head(5)

job ID: 165864b9336c46c4578112d4dfc57da18cd5f78c
job ID: e352cd4ca1469633827f1f7f1860203fb7f72555


,INTERACTION_ID,ENTREZ_GENE_A,ENTREZ_GENE_B,OFFICIAL_SYMBOL_A,OFFICIAL_SYMBOL_B,EXPERIMENTAL_SYSTEM,PUBMED_ID,PUBMED_AUTHOR,THROUGHPUT,QUALIFICATIONS,UniprotID_A,UniprotID_B,UniprotName_A,UniprotName_B
600990,600990,1454,199731,CSNK1E,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-,P49674,Q8NFZ8,KC1E_HUMAN,CADM4_HUMAN
601286,601286,1780,199731,DYNC1I1,CADM4,Two-hybrid,21900206,Vinayagam A (2011),High Throughput,-,O14576,Q8NFZ8,DC1I1_HUMAN,CADM4_HUMAN
1051917,1051917,6472,199731,SHMT2,CADM4,Affinity Capture-RNA,22658674,Castello A (2012),High Throughput,-,P34897,Q8NFZ8,GLYM_HUMAN,CADM4_HUMAN
1196008,1196008,253012,199731,HEPACAM2,CADM4,Affinity Capture-MS,26186194,Huttlin EL (2015),High Throughput,BioPlex 1.0 HEK 293T cells CompPASS score = 0....,A8MVW5,Q8NFZ8,HECA2_HUMAN,CADM4_HUMAN
2232293,2232293,29785,199731,CYP2S1,CADM4,Affinity Capture-MS,28514442,Huttlin EL (2017),High Throughput,BioPlex 2.0 HEK 293T cells CompPASS score = 0....,Q96SQ9,Q8NFZ8,CP2S1_HUMAN,CADM4_HUMAN


In [ ]:
# Converting the UniprotID to Uniprot Name
biogrid_df_filter[['UniprotID_A', 'UniprotID_B']] = biogrid_df_filter[['UniprotID_A', 'UniprotID_B']].map(convert_uniprotID_uniprotName) # takes around 4min

# filter out the columns that are needed
biogrid_df_filter = biogrid_df[['UniprotID_A', 
                                'UniprotID_B', 
                                'EXPERIMENTAL_SYSTEM', 
                                'PUBMED_ID',
                                'PUBMED_AUTHOR']]

# renaming column headers
df_biogrid_final = biogrid_df_filter.rename(columns= {  'UniprotID_A': 'biogrid_interactor_a', 
                                                        'UniprotID_B': 'biogrid_interactor_b',
                                                        'EXPERIMENTAL_SYSTEM': 'biogrid_method',
                                                        'PUBMED_ID': 'biogrid_pubID',
                                                        'PUBMED_AUTHOR': 'biogrid_publication'})

# get one column, only the interactor of the query
df_biogrid = get_interactors_for_target(df_biogrid_final, 'biogrid_interactor_a', 'biogrid_interactor_b', query)

# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_biogrid = removeDuplicateRow_butRetainInfo(df_biogrid, 'interactor_of_' + query ,'biogrid_publication', 'biogrid_method', 'biogrid_pubID' )

df_biogrid.head(5)

In [ ]:
# Extracting the data from the IntAct API using PSCIQUIC

intact_api_url = "http://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices"

version = "current"
method = "interactor"
protein = query
format = "tab25"

intact_request_url = f"{intact_api_url}/{version}/search/{method}/{protein}?format={format}"

intact_raw_data = requests.get(intact_request_url).text

print(intact_raw_data)

In [ ]:
# Getting the data into a pandas df

# Split each row into a list of columns based on PSI-MI TAB 2.5 format
intact_columns = ['Unique identifier for interactor A', 
                    'Unique identifier for interactor B', 
                    'Alternative identifier for interactor A', 
                    'Alternative identifier for interactor B', 
                    'Aliases for A', 
                    'Aliases for B', 
                    'Interaction detection methods', 
                    'First author', 
                    'Identifier of the publication', 
                    'NCBI Taxonomy identifier for interactor A', 
                    'NCBI Taxonomy identifier for interactor B', 
                    'Interaction types', 
                    'Source databases', 
                    'Interaction identifier(s)', 
                    'Confidence score']

intact_rows = [row.split('\t') for row in intact_raw_data.split('\n')]

# Create a pandas DataFrame from the list of rows and columns
intact_df = pd.DataFrame(intact_rows, columns=intact_columns)
intact_df.shape


In [ ]:
# # Cleaning up the dataframe

# filter out the columns that are needed
intact_df_filter = intact_df[['Unique identifier for interactor A', 
                                  'Unique identifier for interactor B', 
                                  'Interaction detection methods', 
                                  'First author', 
                                  'Identifier of the publication', 
                                  'Confidence score']]

# remove the last row (this is a row with no information, an empty row)
intact_df_filter = intact_df_filter[:-1]

# removing the uniprotkbID refix from the name
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(lambda x: x.removeprefix('uniprotkb:'))

# filtering out rows with IntactID instead of UniprotID
intact_df_filter = intact_df_filter[~intact_df_filter['Unique identifier for interactor B'].str.contains('intact:')]

# apply the convert_protein_ID_name function to the first two rows
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(convert_uniprotID_uniprotName) # this step takes a while (around 15min)

In [ ]:
# In the interest of time, I splitted the above cell up

# removing exact duplicate rows
df_intact_dupl = intact_df_filter.drop_duplicates()

# rename the column headers, with prefix IntAct
df_intAct_final = df_intact_dupl.rename(columns= {'Unique identifier for interactor A': 'IntAct_interactor_a', 
                                        'Unique identifier for interactor B': 'IntAct_interactor_b',
                                        'Interaction detection methods': 'IntAct_method',
                                        'First author': 'IntAct_publication',
                                        'Identifier of the publication': 'IntAct_pubID',
                                        'Confidence score': 'IntAct_score'})

# get one column, only the interactor of the query
df_IntAct = get_interactors_for_target(df_intAct_final, 'IntAct_interactor_a', 'IntAct_interactor_b', query)

# clean up the IntAct_score column, that it has only the score and the string ('intact-miscore:')
df_IntAct[['IntAct_score']] = df_IntAct[['IntAct_score']].map(lambda x: x.removeprefix('intact-miscore:'))

# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_IntAct = removeDuplicateRow_butRetainInfo(df_IntAct, 'interactor_of_' + query, 'IntAct_publication', 'IntAct_method', 'IntAct_pubID', 'IntAct_score')

df_IntAct.head(5)

In [ ]:
# Extracting the data from the STRING-API

string_api_url = "https://string-db.org//api"

output_format = "json"
method = "interaction_partners"

string_request_url = "/".join([string_api_url, output_format, method])

params = {
    "identifiers": query,
    "species": 9606, # human
    "required_score": 400,
    "limit": 1000000 
}

response = requests.post(string_request_url, data = params)

string_raw_data = response.text

print(string_raw_data)

In [ ]:
# Getting the STRING-API data into a pandas df
string_df = pd.read_json(string_raw_data)
string_df.head(5)

In [ ]:
# Cleaning up the string dataframe

string_df_filter_2 = string_df

# Converting the GeneID to UniprotID
string_df_filter_2[['UniprotID_A', 'UniprotID_B']] = string_df_filter_2[['preferredName_A', 'preferredName_B']].map(convert_stringID_to_uniprotName) # takes around 11min


In [ ]:
# filter out the columns that are needed
string_df_filter_3 = string_df_filter_2[[  'UniprotID_A', 
                                        'UniprotID_B', 
                                        'score', 
                                        'escore']]

# renaming column headers
df_string_filter_3 = string_df_filter_3.rename(columns= {'UniprotID_A': 'string_interactor_a', 
                                                    'UniprotID_B': 'string_interactor_b',
                                                    'score': 'string_score',
                                                    'escore': 'string_escore'})

# get one column, only the interactor of the query
df_string_1 = get_interactors_for_target(df_string_filter_3, 'string_interactor_a', 'string_interactor_b', query)

print(type(df_string_1))

# removing duplicates but first remove NaN and sort on score
df_string_1 = df_string_1.dropna(subset = ["interactor_of_" + query])
df_string_1 = df_string_1.sort_values(by='string_escore')

# Drop the duplicate rows based on the 'id' and 'age' columns.
df_string = df_string_1.drop_duplicates(subset=["interactor_of_" + query])

# filtering the df on the escore
df_string = df_string[df_string['string_escore'] != 0]

df_string.head(5)

In [ ]:
# # GETTING THE DATA FROM the HIPPIE-API

# hippie_api_url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/queryHIPPIE.php"

# protein_to_query = "CADM4"
# layer = 1 #to query protein within input set (0) or against all HIPPIE proteins (1, default)
# threshold = 0 #confidence threshold, default is 0
# format = "conc_file" #this generates a tab seperated text file (other interesting input types: "mitab", "browser")

# hippie_request_url = f"{hippie_api_url}?proteins={protein_to_query}&layers={layer}&conf_thres={threshold}&out_type={format}"

# hippie_response = requests.get(hippie_request_url).text

# print(hippie_response)

# print(hippie_request_url)

# # there seems to be an issue with the PHP request response, probably the server is not correctly configured

In [ ]:
# ## !!!!! HIPPIE website seems to be down at the moment, worked but is not reliable, got 500 errors

# # GETTING THE DATA FROM HIPPIE through WEBSCRAPING
# import requests
# import json
# import re
# from bs4 import BeautifulSoup

# protein = query

# url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/query.php?s="+str(protein)

# payload = {}
# headers = {
# 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 
# 'Accept-Language': 'nl-NL,nl;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6' ,
# 'Connection': 'keep-alive', 
# 'Referer': 'http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/',
# 'Upgrade-Insecure-Requests': '1' ,
# 'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36' 
# }

# response = requests.request("GET", url, headers=headers, data=payload)
# soup = BeautifulSoup(response.text, "html.parser")
# table = soup.find('tbody') # already skipped the columns names

# # rows = table.find_all('tr')

# # interactions = []
# # for idx, row in enumerate(rows):
# #     data = row.find_all("td")
# #     interaction = {
# #     "Interactor": data[0].text,
# #     "EntrezGeneID": data[1].text,
# #     "GeneSymbol": data[2].text,
# #     "Score": data[3].text
# #     }
# #     interactions.append(interaction) 
   

# print(response)


In [ ]:
# GETTING THE DATA FROM THE APID db by WEBSCRAPING

import requests
import json
import re
from bs4 import BeautifulSoup


def extract_table(proteinid):
    url = "http://cicblade.dep.usal.es:8080/APID/InteractionsGrid.action?protein1="+str(proteinid)+"&protein2=NA"

    payload = {}
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696442421.0.0.0',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/searchProtein.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find('table', id="interactions") #Find the table

    rows = table.find_all("tr") # Find all table rows
    interactions = [] #initialize empty list
    for idx, row in enumerate(rows): #loop over rows, keep index
        if idx == 0: # first row is the header, skip.
            pass
        else:
            try:
                data1 = row.find_all("td") # get column
                interaction = { # build intraction object
                "ProteinA": data1[0].get_text().strip(),
                "ProteinB": data1[1].get_text().strip(),
                "MethodType": data1[2].get_text().strip(),
                "Method": data1[3].get_text().strip(),
                "Publication": re.sub(' +', ' ',data1[4].get_text().strip().replace("\n", "")),
                "Source": data1[5].get_text().strip()
                }
                interactions.append(interaction) #append interaction object to result list
            except:
                pass
    return interactions #return the result list


# To find the UniProtID from the protein name
proteinid = convert_protein_ID_Name(query)
apid_results = extract_table(proteinid)
# print(json.dumps(apid_results, indent=4))
print(apid_results)

In [ ]:
# Getting the data from APID into a df

apid_df = pd.DataFrame(apid_results)

# filter out the columns that are needed
apid_df_filter = apid_df[[  'ProteinA', 
                                'ProteinB', 
                                'Method', 
                                'Publication',
                                'Source']]

# renaming column headers
df_apid_final = apid_df_filter.rename(columns= {'ProteinA': 'apid_interactor_a', 
                                                    'ProteinB': 'apid_interactor_b',
                                                    'Method': 'apid_method',
                                                    'Publication': 'apid_publication',
                                                    'Source': 'apid_source'})

# get one column, only the interactor of the query
df_apid_int = get_interactors_for_target(df_apid_final, 'apid_interactor_a', 'apid_interactor_b', query)


# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_apid = removeDuplicateRow_butRetainInfo(df_apid_int, 'interactor_of_' + query, 'apid_method', 'apid_publication', 'apid_source')

df_apid.head(5)

In [ ]:
# Make an intersecions diagram using the pyUpSet

# Put the data into the right format using the custom function
biogrid_interactors = convert_column_to_list(df_biogrid, 'interactor_of_' + query)
IntAct_interactors = convert_column_to_list(df_IntAct, 'interactor_of_' + query)
string_interactors = convert_column_to_list(df_string, 'interactor_of_' + query)
apid_interactors = convert_column_to_list(df_apid, 'interactor_of_' + query)

# Plot the data into an upSetplot
ppis = from_contents({'BioGrid': biogrid_interactors, 'IntAct': IntAct_interactors, 'STRING': string_interactors, 'APID': apid_interactors})
ax_dict = UpSet(ppis, subset_size='count', show_counts=True).plot()

In [ ]:
import collections

list1 = [1, 2, 3, 4, 5, 3, 4]

# Check for duplicates
counter = collections.Counter(string_interactors)
if any(count > 1 for count in counter.values()):
    print("The list contains duplicates.")
else:
    print("The list does not contain duplicates.")

# Extract the duplicate values
duplicate_values = [key for key, count in counter.items() if count > 1]
print(duplicate_values)

print(string_interactors)

# STRING has 2 duplicates

In [ ]:
# merge the separate dataframes together based on one column
dfs = [df_biogrid, df_IntAct, df_string, df_apid]
final_df = reduce(lambda left, right: pd.merge(left,right, on=['interactor_of_' + query], how='outer'), dfs)

In [ ]:
# filter the dataframe based on the subcellular location, using the Uniprot-API

def get_subcellular_location(protein):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{protein}?format={format}"
    uniprot_response = requests.get(uniprot_request_url)

    uniprot_json = uniprot_response.json()

    subcellular_locations = []
    for comment in uniprot_json.get('comments', []):
        if comment.get('commentType') == 'SUBCELLULAR LOCATION':
            for subcellular_location in comment.get('subcellularLocations', []):
                location_value = subcellular_location.get('location', {}).get('value', '')
                subcellular_locations.append(location_value)
    
    return subcellular_locations

# make a column with the UniprotID from the interactor, this will be used for the subcellular location
# final_df[['UniprotID']] = final_df[['interactor_of_' + query]].map(convert_protein_ID_Name)

# # now go over the dataframe and make a new column with the subcellular location
final_df[['subcellularLocation']] = final_df[['interactor_of_' + query]].map(get_subcellular_location) # takes around 5min


In [ ]:
# filter on the following subcellular locations
subcellular_locations = ["Membrane", "membrane",
                         "Cell junction", "cell junction",
                         "Cell projection", "cell projection",
                         "Cell membrane", "cell membrane",
                         "Plasma membrane", "plasma membrane",
                         "Secreted", "secreted",
                         "Extracellular space", "extracellular space",
                         "Extracellular matrix", "extracellular matrix"
                         "Extracellular exosome", "extracellular exosome",
                         "Cell surface", "cell surface"]

final_df_filtered = final_df[final_df['subcellularLocation'].apply(lambda x: any(location in subcellular_locations for location in x))]

# exporting to a csv
final_df_filtered.to_csv("interactors_of_" + query + ".csv")

In [ ]:
# Add the gene symbols & geneID (now I'm using mouse genes because there are more GO terms for these then human )

# first convert the human uniprotName to human uniprotID, make new column "interactor_of_VCAM1_uniprotID"
final_df_filtered[["interactor_of_" + query + "_uniprotID"]] = final_df_filtered[['interactor_of_' + query]].map(convert_protein_ID_Name)




In [ ]:
# then convert the human uniprotID to human gene symbol
def convert_uniprotID_geneSymbol(UniprotID):
    mygene_api_url = "https://mygene.info/v3/query?q="
    UniprotID = UniprotID
    mygene_request_url =f"{mygene_api_url}{UniprotID}&species=human"
 
    mygene_response = requests.get(mygene_request_url)
    mygene_json = mygene_response.json()   
    
    if "hits" in mygene_json and len(mygene_json["hits"]) > 0 and "symbol" in mygene_json["hits"][0]:
        gene_symbol = mygene_json["hits"][0]["symbol"]
        return gene_symbol
    else:
        return None

# example:
print(convert_uniprotID_geneSymbol("Q13797"))

# now make a new column "interactor_of_VCAM1_HUMAN_geneSymbol"
final_df_filtered[["interactor_of_" + query + "_geneSymbol"]] = final_df_filtered[["interactor_of_" + query + "_uniprotID"]].map(convert_uniprotID_geneSymbol)


In [ ]:
# exporting to a csv
final_df_filtered.to_csv("interactors_of_" + query + ".csv")